## Metadata Query Agent — Server-Side Batch Evaluation (Ground Truth)

Evaluates the **deployed** Metadata Query Agent (`MetadataQueryRuntimeArn`) **entirely server-side** with Amazon Bedrock **AgentCore Batch Evaluations** (`bedrock_agentcore.evaluation.BatchEvaluationRunner`). The only client-side work is invoking the agent once per ground-truth row to produce spans, and reading the response payload to record cost/latency.

### Why SESSION-level evaluators
This agent can answer in **one** turn, or ask a **clarifying question** first and answer on a later turn. A SESSION-level evaluator reads the whole conversation via `{context}` and scores the **final** answer once — the right granularity for a multi-turn agent. Per-turn `Builtin.Correctness` / `Builtin.Helpfulness` builtins are not used: a per-turn judge errors on a clarification turn (no model/tool call) and answer quality is a per-conversation question.

### What gets scored (all server-side, all SESSION-level custom LLM-as-Judge evaluators)
Created with `create_evaluator`, scored in-service — no Lambda:

| Evaluator | Level | Placeholders | Checks |
|:--|:--|:--|:--|
| `GoalSuccess` | SESSION | `{context}`, `{assertions}` | The agent achieved the user's goal across the conversation (path + final-answer assertion). Custom SESSION judge that replaced the un-editable `Builtin.GoalSuccessRate`, which mis-scored clarification turns by grading an intermediate graph span |
| `FinalAnswerFaithfulness` | SESSION | `{context}`, `{assertions}` | The conversation's FINAL answer matches the expected answer (threaded into `{assertions}`) |
| `SqlGrounded` | SESSION | `{context}`, `{available_tools}` | Every table/column in the executed SQL is present in the retrieved schema context — Phase 1 KB chunks / Phase 3 slice |

> **Placeholder constraint.** A SESSION evaluator may reference only `{context}`, `{available_tools}`, `{assertions}`, `{expected_tool_trajectory}`, `{actual_tool_trajectory}`. `{expected_response}` is **TRACE-only** — so the expected final answer is threaded in as a SESSION assertion (`eval_multiturn.build_trajectory_assertions`, prefixed `FINAL_ANSWER_ASSERTION_PREFIX`) and `FinalAnswerFaithfulness` reads it from `{assertions}`.

### Metrics captured per query
The batch result exposes only the **evaluators'** token usage, never the **agent's**. The `agent_invoker` records, per turn, the query agent's `usage` (input/output/total tokens), `runtimeMs`, and wall-clock latency into `AGENT_RUNS`.

### Prerequisites
- **Notebook 1** run to `%store metadata_id`
- **AgentCore stack** deployed; Metadata Query Agent running
- `bedrock-agentcore>=1.11.0`

In [1]:
# Prereq (uncomment if not already installed):
# !pip install bedrock-agentcore-starter-toolkit==0.3.9

## 1. Setup & Credentials

In [2]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Load .env if present, then set defaults
load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
print(f"✓ AWS Credentials Verified:")
print(f"  Profile: {os.environ['AWS_PROFILE']}")
print(f"  Account: ...{identity['Account'][-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region: {region}")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


✓ AWS Credentials Verified:
  Profile: huthmac+demo
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1


In [3]:
import re

def _redact_account_ids(obj):
    """Recursively mask AWS account IDs in ARN strings to their last 4 digits (********NNNN)."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(\d{8})(\d{4})', r'********\2', obj)
    return obj


def _mask_acct(text):
    """Mask AWS account IDs (12-digit runs) to their last 4 digits — e.g. ********2087 —
    in any string (ARNs, bucket names, plain ids). Never emits the full account number."""
    import re as __re
    if not isinstance(text, str):
        return text
    return __re.sub(r'(\d{8})(\d{4})', r'********\2', text)


In [4]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# Invoke the JWT-inbound runtime with a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    cfg = lam.get_function_configuration(FunctionName=f'{PROJECT_NAME}-mcp-tools')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


## 2. Load the Groundtruth Dataset

Schema (one record per row):

| Field | Type | Description |
|:------|:-----|:------------|
| `Natural_Language_Question` | string | The question fed to the agent |
| `Expected_Answer` | string | The natural-language answer the agent should produce |
| `Expected_SQL_Query` | string | The SQL query that would correctly answer the question |
| `Expected_SQL_Result` | list[dict] | The result rows that SQL should return |

The dataset drives the whole run: one scenario per row is built (section 5), the agent is
invoked once per scenario, and the groundtruth fields (the expected answer + expected
SQL/result) are threaded into each scenario's `assertions` so the server-side SESSION custom
evaluators (`GoalSuccess`, `FinalAnswerFaithfulness`, `SqlGrounded`) can read them.

In [5]:
with open('../data/eval/groundtruth_dataset.json', 'r') as f:
    groundtruth = json.load(f)

# Data-query rows need SQL + expected result; advisory/meta rows (Expected_Tier
# == 'advisory') are answered by the intent-router advisory path and carry only
# a question + Expected_Answer, so they are exempt from the SQL requirement.
BASE_COLS = {'Natural_Language_Question', 'Expected_Answer'}
SQL_COLS = {'Expected_SQL_Query', 'Expected_SQL_Result'}
for i, row in enumerate(groundtruth):
    is_advisory = str(row.get('Expected_Tier', '')).lower() == 'advisory'
    required = BASE_COLS if is_advisory else (BASE_COLS | SQL_COLS)
    missing = required - set(row.keys())
    if missing:
        raise ValueError(f"Row {i} missing columns: {missing}")

# Read-only knob from the environment (see notebooks/.env) — never set here.
MAX_ROWS = int(os.environ.get('MAX_ROWS', '0'))
if MAX_ROWS > 0:
    groundtruth = groundtruth[:MAX_ROWS]
    print(f"⚠ MAX_ROWS={MAX_ROWS} — evaluating a {len(groundtruth)}-row slice")

df_gt = pd.DataFrame(groundtruth)

# ── Loud ground-truth quality check ──────────────────────────────────────────
# The custom SESSION judges (GoalSuccess, FinalAnswerFaithfulness, SqlGrounded) score the agent against
# Expected_Answer / Expected_SQL_Query / assertions.
def _is_placeholder(v) -> bool:
    """True when a ground-truth field is empty or the literal 'PLACEHOLDER' sentinel."""
    return v in (None, '', [], 'PLACEHOLDER')

_ph_sql = sum(1 for r in groundtruth if _is_placeholder(r.get('Expected_SQL_Query')))
_ph_ans = sum(1 for r in groundtruth if _is_placeholder(r.get('Expected_Answer')))
print(f"✓ Loaded {len(df_gt)} groundtruth row(s)")
if _ph_sql or _ph_ans:
    print(f"  ⚠ {_ph_sql}/{len(groundtruth)} rows have a PLACEHOLDER Expected_SQL_Query and "
          f"{_ph_ans}/{len(groundtruth)} have a PLACEHOLDER Expected_Answer.")
    print(f"    The custom GoalSuccess / FinalAnswerFaithfulness judges will score "
          f"against placeholders for those rows — fill them in for meaningful scores.")
display(df_gt[['Natural_Language_Question', 'Expected_SQL_Query']])
# @multiturn-cell7  Multi-turn support: import the shared helpers (pure, no I/O)
# and report which rows are multi-turn so the run summary is honest.
try:
    from agents.shared.eval_multiturn import parse_multiturn_row
except ImportError:
    import sys
    sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import parse_multiturn_row
_specs = [parse_multiturn_row(r, index=i) for i, r in enumerate(groundtruth)]
_mt = [s for s in _specs if len(s['turns']) > 1 or s['mode'] != 'scripted']
print(f"✓ {len(_specs)} scenario(s): {len(_mt)} multi-turn, {len(_specs) - len(_mt)} single-turn")


✓ Loaded 16 groundtruth row(s)
  ⚠ 3/16 rows have a PLACEHOLDER Expected_SQL_Query and 0/16 have a PLACEHOLDER Expected_Answer.
    The custom GoalSuccess / FinalAnswerFaithfulness judges will score against placeholders for those rows — fill them in for meaningful scores.


,Natural_Language_Question,Expected_SQL_Query
0,Show me policies where the insured party is also the policyholder.,"SELECT DISTINCT h.holding_id, h.policy_id, lp.party_id AS insured_party_id\nFROM normalized.holding h\nJOIN normalized.life_participant lp \n ON lp.holding_id = h.holding_id\nJOIN (\n SELECT holding_id, party_id\n FROM normalized.life_participant\n WHERE is_deleted = false\n GROUP BY holding_id, party_id\n HAVING COUNT(DISTINCT participant_sk) > 1\n) dup \n ON dup.holding_id = h.holding_id \n AND dup.party_id = lp.party_id\nWHERE lp.is_deleted = false \n AND h.is_deleted = false"
1,"For each rider, who are the insured participants and what are their roles?","SELECT\n r.rider_id,\n r.rider_name,\n r.rider_code,\n r.rider_type,\n r.rider_status,\n r.holding_id,\n c.coverage_id,\n c.coverage_type AS participant_role,\n c.party_id,\n COALESCE(NULLIF(p.full_name, ''), CONCAT(p.first_name, ' ', p.last_name)) AS participant_name,\n p.party_type,\n c.issue_age,\n c.issue_gender,\n c.underwriting_class\nFROM normalized.rider r\nJOIN normalized.coverage c\n ON r.holding_id = c.holding_id\nLEFT JOIN normalized.party p\n ON c.party_id = p.party_id\nWHERE r.is_deleted = false\n AND c.is_deleted = false\nORDER BY r.rider_id, c.coverage_id\nLIMIT 10"
2,List the top 5 most common party types and their human-readable descriptions.,"SELECT p.party_type, COUNT(*) AS party_count\nFROM normalized.party p\nWHERE p.is_deleted = false\nGROUP BY p.party_type\nORDER BY party_count DESC\nLIMIT 5"
3,"What is the total market value of all active holdings, grouped by party?","SELECT\n p.party_id,\n p.full_name AS party_name,\n COUNT(DISTINCT h.holding_id) AS holding_count,\n SUM(h.market_value) AS total_market_value\nFROM normalized.holding h\nJOIN normalized.coverage c\n ON h.holding_id = c.holding_id\nJOIN normalized.party p\n ON CONCAT('PARTY#', c.party_id) = p.party_id\nWHERE h.holding_status = 'Active'\n AND h.is_deleted = false\n AND c.is_deleted = false\n AND p.is_deleted = false\nGROUP BY p.party_id, p.full_name\nORDER BY total_market_value DESC\nLIMIT 10"
4,"For policies that have a payout schedule, show the policyholder's name, the payout frequency, and the projected next-payout amount.","SELECT\n p.full_name AS policyholder_name,\n COALESCE(pp.premium_mode, c.product_code) AS payout_frequency,\n fa.transaction_amount AS projected_next_payout_amount,\n fa.effective_date AS payout_effective_date,\n fa.activity_type AS payout_type,\n ad.holding_id AS policy_id\nFROM normalized.annuity_detail ad\nJOIN normalized.coverage c\n ON c.holding_id = ad.holding_id\n AND c.is_deleted = false\nJOIN normalized.party p\n ON p.party_id = CONCAT('PARTY#', c.party_id)\n AND p.is_deleted = false\nLEFT JOIN normalized.policy_product pp\n ON pp.product_code = c.product_code\n AND pp.is_deleted = false\nLEFT JOIN normalized.financial_activity fa\n ON fa.holding_id = ad.holding_id\n AND fa.activity_type IN ('Withdrawal', 'Dividend', 'Claim')\n AND fa.is_deleted = false\nWHERE ad.deleted = false\nORDER BY ad.holding_id, fa.effective_date DESC\nLIMIT 10"
5,How many parties are there?,SELECT COUNT(*) AS party_count FROM normalized.party WHERE is_deleted = false
6,List the top 10 coverage products by name.,SELECT product_name FROM normalized.coverage_product WHERE is_deleted = false ORDER BY product_name ASC LIMIT 10
7,"Show the top 10 parties by total holding market value, including the investment product names they hold.","SELECT\n p.party_id,\n p.full_name AS party_name,\n SUM(h.market_value) AS total_market_value,\n ARRAY_JOIN(ARRAY_AGG(DISTINCT\n CASE\n WHEN hs.fundname IS NOT NULL THEN hs.fundname\n WHEN pp.product_name IS NOT NULL THEN pp.product_name\n ELSE 'N/A'\n END\n ), ', ') AS investment_products\nFROM normalized.party p\nJOIN normalized.coverage c\n ON c.party_id = REPLACE(p.party_id, 'PARTY#', '')\n AND c.is_deleted = false\nJOIN normalized.holding h\n ON h.holding_id = c.holding_id\n AND h.is_deleted = false\nLEFT JOIN normalized.holding_suba

✓ 16 scenario(s): 3 multi-turn, 13 single-turn


## 3. Resolve the AgentCore Runtime

Restore `metadata_id` (stored by notebook 1) and look up the Metadata Query Runtime
ARN from the AgentCore CloudFormation stack outputs.

In [6]:
# ── Resolve the semantic-rag layer to evaluate ──
# Prefer the id stored by notebook 1 (%store), but VALIDATE it against the
# metadata table — a stored id from an earlier deploy can be stale (the layer
# was rebuilt under a new id), which makes the agent retrieve 0 KB sources and
# the batch FAIL. If the stored id is missing or no longer present/completed,
# fall back to the most-recently-updated COMPLETED semantic-rag layer.
_meta_table = session.resource('dynamodb', region_name=region).Table(
    os.environ.get('ONTOLOGY_METADATA_TABLE', f'{PROJECT_NAME}-metadata')
)


def _layer_ok(layer_id: str) -> bool:
    """True when ``layer_id`` exists in the metadata table with status=completed."""
    if not layer_id:
        return False
    resp = _meta_table.query(
        KeyConditionExpression='id = :i',
        ExpressionAttributeValues={':i': layer_id},
    )
    return any(it.get('status') == 'completed' for it in resp.get('Items', []))


def _latest_completed_semantic_rag() -> str:
    """Return the newest completed ``semantic-rag-*`` layer id (or '' if none)."""
    items, scan_kw = [], {
        'FilterExpression': 'contains(id, :p) AND #s = :c',
        'ExpressionAttributeNames': {'#s': 'status'},
        'ExpressionAttributeValues': {':p': 'semantic-rag', ':c': 'completed'},
    }
    while True:
        page = _meta_table.scan(**scan_kw)
        items.extend(page.get('Items', []))
        if 'LastEvaluatedKey' not in page:
            break
        scan_kw['ExclusiveStartKey'] = page['LastEvaluatedKey']
    if not items:
        return ''
    items.sort(key=lambda it: it.get('updatedAt') or it.get('createdAt') or '',
               reverse=True)
    return items[0]['id']


# Restore the stored id (no-op if notebook 1 hasn't run this session).
metadata_id = ''
try:
    %store -r metadata_id
except Exception:
    metadata_id = ''

EVAL_ID = metadata_id
if _layer_ok(EVAL_ID):
    print(f"✓ Using EVAL_ID from %store (validated completed): {EVAL_ID}")
else:
    fallback = _latest_completed_semantic_rag()
    if not fallback:
        raise RuntimeError(
            "No completed semantic-rag layer found in the metadata table. "
            "Build a Semantic RAG layer (notebook 1 / the admin UI) first."
        )
    print(f"⚠ Stored metadata_id ({EVAL_ID!r}) is stale/missing — falling back "
          f"to the latest completed semantic-rag layer.")
    EVAL_ID = fallback
    metadata_id = EVAL_ID
    # Refresh the store so downstream notebooks agree (the %store magic must be
    # its own statement — an inline comment after it is mis-parsed by IPython).
    %store metadata_id
    print(f"✓ Using EVAL_ID: {EVAL_ID}")

# ── Look up Metadata Query Runtime ARN + derive agent_id ──
cfn = session.client('cloudformation', region_name=region)
stack_name = f'{PROJECT_NAME}-agentcore'
outputs = cfn.describe_stacks(StackName=stack_name)['Stacks'][0]['Outputs']
output_map = {o['OutputKey']: o['OutputValue'] for o in outputs}

METADATA_QUERY_RUNTIME_ARN = output_map['MetadataQueryRuntimeArn']
# arn:aws:bedrock-agentcore:region:account:runtime/<agent_id>
agent_id = METADATA_QUERY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"\n✓ AgentCore Runtime (from stack '{stack_name}'):")
print(f"  Runtime ARN: {_mask_acct(METADATA_QUERY_RUNTIME_ARN)}")
print(f"  Agent ID:    {agent_id}")

✓ Using EVAL_ID from %store (validated completed): semantic-rag-multi_table_curated_layer-7e509128

✓ AgentCore Runtime (from stack 'semantic-layer-dev-agentcore'):
  Runtime ARN: arn:aws:bedrock-agentcore:us-east-1:********4087:runtime/semantic_layer_dev_metadata_query-6aPZJf6eWO
  Agent ID:    semantic_layer_dev_metadata_query-6aPZJf6eWO


## 4. Create Custom (LLM-as-Judge) Evaluators

All three custom evaluators are **SESSION-level** — each reads the full conversation via `{context}` across all turns, scoring the final answer once. This is the correct granularity for a multi-turn agent: a clarify-then-answer conversation is judged on its destination, not on each intermediate turn.

| Evaluator | Level | Placeholders | Checks |
|:--|:--|:--|:--|
| `FinalAnswerFaithfulness` | SESSION | `{context}`, `{assertions}` | The conversation's FINAL answer matches the expected answer |
| `SqlGrounded` | SESSION | `{context}`, `{available_tools}` | Every table/column in the executed SQL appears in the retrieved schema context — KB chunks / slice |

> **How `FinalAnswerFaithfulness` gets the expected answer.** SESSION evaluators cannot use `{expected_response}` (TRACE-only). The expected answer is threaded into `{assertions}` as a line prefixed `The conversation's final answer matches: …` (by `eval_multiturn.build_trajectory_assertions`), and the judge extracts it from there.

> **How `SqlGrounded` sees the SQL and schema without a Lambda.** The deployed agent runs a deterministic graph (phase fns), so `{context}` (built from the session spans) carries the Phase 1 KB chunks / Phase 3 schema slice and the `execute_sql_query` tool **call arguments** (the SQL). The judge reads both from `{context}` — no Lambda evaluator needed.

> Re-running this cell creates **new** evaluators (unique name suffix). Delete old ones via `delete_evaluator` if you iterate often.


In [7]:
# Custom SESSION LLM-as-Judge evaluators — defined ONCE in
# agents/shared/eval_judges.py and imported here so all notebooks score with
# byte-identical judges (no inline prompt drift). See that module for why the
# judges are SESSION-level and why RAG vs VKG keep separate prompt families.
try:
    from agents.shared.eval_judges import create_custom_judges, JUDGE_MODEL_ID
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_judges import create_custom_judges, JUDGE_MODEL_ID

_cp = session.client('bedrock-agentcore-control', region_name=region)

# family='rag' selects the Semantic-RAG / metadata_query prompt set. Returns the ids ORDERED
# [GoalSuccess, FinalAnswerFaithfulness, SqlGrounded]. GoalSuccess is a custom SESSION judge
# used instead of the un-editable Builtin.GoalSuccessRate (its instructions can't be changed, so
# on this deterministic-graph agent it mistook an intermediate intent-classification JSON span
# for the answer and scored correctly-handled clarification conversations 0.0).
GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID = create_custom_judges(
    control_client=_cp, family='rag',
)

CUSTOM_EVALUATOR_IDS = [GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID]
print("✓ Custom server-side evaluators created (all SESSION-level, LLM-as-Judge, no Lambda):")
print(f"  GoalSuccess             (SESSION) : {GOAL_SUCCESS_ID}")
print(f"  FinalAnswerFaithfulness (SESSION) : {ANSWER_FAITHFUL_ID}")
print(f"  SqlGrounded             (SESSION) : {SQL_GROUNDED_ID}")


✓ Custom server-side evaluators created (all SESSION-level, LLM-as-Judge, no Lambda):
  GoalSuccess             (SESSION) : GoalSuccess_847b3df5-0EcbCECnsF
  FinalAnswerFaithfulness (SESSION) : FinalAnswerFaithfulness_847b3df5-4RJPHnAUIW
  SqlGrounded             (SESSION) : SqlGrounded_847b3df5-ANxPbNGHSa


## 5. Build the Dataset (one scenario per row) + agent_invoker

**Per-row scenarios** — each ground-truth question is its own `PredefinedScenario` with its own
session, so we get **per-query** evaluator scores (via `fetch_evaluation_events`) and clean
**per-query** agent token/latency rows.

The `agent_invoker` is the only client-side work: it invokes the synchronous query agent once
per scenario and records the agent's own cost/latency (from `metadata.usage` / `runtimeMs` and
wall-clock) into `AGENT_RUNS`, keyed by the framework session id. Its return value
(`agent_output`) is the answer text the TRACE evaluators score.


In [8]:
# @multiturn-cell13  Chat-stream invoker (Option A) + multi-turn dataset builder.
import json, time
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput)
from bedrock_agentcore.evaluation.runner.dataset_types import Dataset
try:
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)

# Detect the simulation extra once; gate simulated scenarios on it.
try:
    import strands_evals  # noqa: F401
    SIMULATED_ENABLED = True
except ImportError:
    SIMULATED_ENABLED = False
print(f"simulated mode: {'ENABLED' if SIMULATED_ENABLED else 'DISABLED (scripted only)'}")

AGENT_RUNS = {}              # keyed by run_key(session_id, turn_idx)

def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Drive the agent CHAT-STREAM path so it reads+persists history itself.

    The SDK reuses one session_id across a scenario's turns; the chat entrypoint
    keys DDB history off the payload sessionId, so turn N+1 resolves turn N's
    clarification exactly as in production. We parse the SSE response, fold any
    clarification option labels into the judge-visible output, and record
    per-turn cost/latency keyed by (session_id, turn_idx).
    """
    sid = invoker_input.session_id
    # turn index derived from AGENT_RUNS so a notebook re-run (AGENT_RUNS reset above) is safe
    turn_idx = sum(1 for k in AGENT_RUNS if k.startswith(f"{sid}#turn"))
    message = (invoker_input.payload if isinstance(invoker_input.payload, str)
               else json.dumps(invoker_input.payload))
    payload = build_chat_payload(message=message, session_id=sid,
                                 ontology_id=EVAL_ID, turn_idx=turn_idx)
    start = time.time(); err = None; parsed = {}
    try:
        raw = _invoke_runtime(METADATA_QUERY_RUNTIME_ARN, sid,
                              json.dumps(payload).encode("utf-8"))
        parsed = parse_chat_stream_sse(raw.decode("utf-8", errors="replace"))
        err = parsed.get("error")
    except Exception as exc:  # noqa: BLE001
        err = str(exc); print(f"  ⚠ [{sid} t{turn_idx}] {err}")
    _rows = parsed.get("rows", []) or []
    AGENT_RUNS[run_key(sid, turn_idx)] = {
        "scenario_session": sid, "turn_idx": turn_idx, "message": message,
        "answer": parsed.get("answer", ""), "agent_sql": parsed.get("sql", ""),
        # Executed-vs-generated SQL + degrade reason + the actual result rows, so
        # the result file is self-contained evidence (NL question + SQL + result).
        "executed_sql": parsed.get("executed_sql", ""),
        "degraded": parsed.get("degraded"),
        "clarified": bool(parsed.get("clarification")),
        "rows": _rows, "result_row_count": len(_rows), "result_sample": _rows[:3],
        "usage": parsed.get("usage", {}),
        "runtime_ms": parsed.get("runtime_ms"),
        # Provenance tier (Step 5) — read by the correct-tier assertion in cell 17.
        "provenance": parsed.get("provenance") or {},
        "wall_clock_s": round(time.time() - start, 2), "invoke_error": err,
    }
    state = "clarify" if parsed.get("clarification") else ("answer" if parsed.get("sql") else "none")
    print(f"  {'OK' if err is None else 'XX'} [{sid} t{turn_idx}] "
          f"{AGENT_RUNS[run_key(sid, turn_idx)]['wall_clock_s']}s . {state}")
    return AgentInvokerOutput(agent_output=format_agent_output(parsed) or (err or ""))

_specs = [parse_multiturn_row(r, index=i) for i, r in enumerate(groundtruth)]
dataset = Dataset(scenarios=build_scenarios(
    _specs, ontology_id=EVAL_ID, simulated_enabled=SIMULATED_ENABLED))
EXPECTED_TRAJECTORY = ["execute_sql_query"]
print(f"✓ Dataset: {len(dataset.scenarios)} scenario(s) "
      f"({sum(1 for s in _specs if len(s['turns'])>1)} multi-turn)")


simulated mode: ENABLED
✓ Dataset: 16 scenario(s) (2 multi-turn)


## 6. Run the Batch Evaluation (server-side)

`run_dataset_evaluation` invokes the agent for each scenario (via `agent_invoker`), waits
`ingestion_delay_seconds` for CloudWatch span ingestion, submits a single `StartBatchEvaluation`
job over the **SESSION-only custom** evaluators, and polls to completion. All scoring
happens in-service. Span discovery is by **service name + session id + time range**.

In [9]:
# @multiturn-cell15  k-run batch runner: SESSION-only evaluator set (clarification-safe).
# A single batch is a noisy draw (LLM-judge + agent stochasticity). We run the WHOLE batch
# EVAL_K times (default 3) and AVERAGE the per-evaluator aggregate scores in cell 17, so the
# headline numbers are robust. Each run clears + repopulates AGENT_RUNS via agent_invoker and
# snapshots that run's aggregate scores, per-query events, agent_runs, and cost/latency.
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)
try:
    from agents.shared.eval_multiturn import group_runs_by_session
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import group_runs_by_session

SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    # Multi-turn sessions emit more spans, and clarification turns ingest a beat later;
    # a longer wait gives short clarify turns time to ingest their answer span before scoring.
    ingestion_delay_seconds=420,
)

# ── SESSION-ONLY evaluator set ───────────────────────────────────────────────
# SESSION judges read the full conversation via {context} and score the final answer once.
# (Clarify turns now always emit an answer span via shared/answer_span.emit_answer_span, so the
# old "TRACE errors on clarify turns → whole session FAILED" failure no longer occurs; we still
# drop the per-turn TRACE builtins Builtin.Correctness/Helpfulness because answer quality is a
# per-conversation question.) GoalSuccess is now a CUSTOM SESSION judge (not Builtin) — see the
# evaluator cell for why: the un-editable Builtin.GoalSuccessRate grades an intermediate graph
# span and failed correctly-handled clarification conversations.
#   - GoalSuccess              → did the agent achieve the user's goal (path + final answer)
#   - FinalAnswerFaithfulness  → answer quality (SESSION-level, uses {context})
#   - SqlGrounded              → no hallucinated schema in executed SQL
ALL_EVALUATOR_IDS = [
    *CUSTOM_EVALUATOR_IDS,            # SESSION — GoalSuccess, FinalAnswerFaithfulness, SqlGrounded
]

# Map evaluator id → display name for the per-run aggregate scores. Each custom evaluator
# maps its UUID to a clean name (GoalSuccess, used instead of 'Builtin.GoalSuccessRate').
_EVAL_NAME = {GOAL_SUCCESS_ID: 'GoalSuccess',
              ANSWER_FAITHFUL_ID: 'FinalAnswerFaithfulness',
              SQL_GROUNDED_ID: 'SqlGrounded'}
# AgentCore eval EVENTS carry gen_ai.evaluation.name = the evaluator NAME (the id
# minus its trailing '-<hash>' segment), NOT the full evaluatorId. Alias those names
# to the clean display name so per-query rows + per_scenario_goal_mean resolve
# correctly (else they fall through to the hashed string and the 'GoalSuccess' filter
# misses, leaving per_scenario_goal_mean empty).
_EVAL_NAME.update({_eid.rsplit('-', 1)[0]: _disp for _eid, _disp in list(_EVAL_NAME.items())})

# k-run repetition knob (read-only, from notebooks/.env). EVAL_K=1 → single run (old behaviour).
EVAL_K = int(os.environ.get('EVAL_K', '3'))
batch_runner = BatchEvaluationRunner(region=region)


def _scenario_id_for_session(sess):
    """Map a framework session id (scenario_id + '-' + uuid) back to its scenario_id."""
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''


def _aggregate_scores(result) -> dict:
    """Pull the per-evaluator average scores out of a finished batch result.

    Parameters:
        result: the BatchEvaluationRunner result for one run.
    Returns:
        {display_name: average_score|None} over ALL_EVALUATOR_IDS.
    """
    scores = {}
    ev = result.evaluation_results
    if ev is not None:
        for es in (ev.evaluator_summaries or []):
            st = es.statistics
            name = _EVAL_NAME.get(es.evaluator_id, es.evaluator_id)
            scores[name] = (round(st.average_score, 4)
                            if st and st.average_score is not None else None)
    return scores


def _agent_perf_snapshot() -> dict:
    """Summarise the CURRENT AGENT_RUNS into one run's cost/latency totals."""
    lat = [r['wall_clock_s'] for r in AGENT_RUNS.values() if r.get('wall_clock_s') is not None]

    def _usum(key: str) -> int:
        return sum(int((r.get('usage') or {}).get(key) or 0) for r in AGENT_RUNS.values())

    return {'turns': len(AGENT_RUNS),
            'avg_wall_clock_s': round(sum(lat) / len(lat), 2) if lat else None,
            'input_tokens': _usum('inputTokens'), 'output_tokens': _usum('outputTokens'),
            'total_tokens': _usum('totalTokens')}


def _fetch_events(result) -> list:
    """Fetch this run's per-query evaluator events (output stream lags COMPLETED — retry)."""
    for _ in range(6):
        try:
            evs = batch_runner.fetch_evaluation_events(result)
            if evs:
                return evs
        except (LookupError, ValueError):
            pass
        time.sleep(20)
    return []


def _event_rows_for_run(result) -> list:
    """Flatten this run's evaluator events into per-(scenario, evaluator) score rows."""
    grouped = group_runs_by_session(AGENT_RUNS)

    def _f(e, key):
        return e[key] if key in e else (e.get('attributes', {}) or {}).get(key)

    rows = []
    for e in _fetch_events(result):
        sess = _f(e, 'session.id') or _f(e, 'gen_ai.session.id')
        turns = grouped.get(sess, [])
        name = _f(e, 'gen_ai.evaluation.name')
        rows.append({
            'scenario_id': _scenario_id_for_session(sess),
            'question': (turns[0]['message'] if turns else ''),
            'evaluator': _EVAL_NAME.get(name, name),
            'score': _f(e, 'gen_ai.evaluation.score.value'),
            'label': _f(e, 'gen_ai.evaluation.score.label'),
        })
    return rows


def _run_one_batch(run_idx: int) -> dict:
    """Run the SESSION-only batch once (fresh AGENT_RUNS); return a per-run snapshot dict."""
    AGENT_RUNS.clear()  # agent_invoker repopulates this per turn for THIS run
    cfg = BatchEvaluationRunConfig(
        batch_evaluation_name=f"query_gt_batch_r{run_idx}_{uuid.uuid4().hex[:8]}",
        description=f"Server-side SESSION-only ground-truth eval of the metadata query agent "
                    f"(clarification-safe), run {run_idx}/{EVAL_K}.",
        evaluator_config=BatchEvaluatorConfig(evaluator_ids=ALL_EVALUATOR_IDS),
        data_source=batch_data_source,
        max_concurrent_scenarios=3,   # synchronous agent, up to 900s/row
        # k=3 x 16 rows can approach a 3600s ceiling on a slow draw; 5400s
        # keeps a full k-run from being marked incomplete (matches nb5).
        polling_timeout_seconds=5400,
        polling_interval_seconds=30,
    )
    print(f"\n=== Run {run_idx}/{EVAL_K}: {cfg.batch_evaluation_name} ===")
    res = batch_runner.run_dataset_evaluation(
        config=cfg, dataset=dataset, agent_invoker=agent_invoker)
    print(f"  ✓ status: {res.status}")
    if res.agent_invocation_failures:
        print(f"  ⚠ Agent invocation failures: {len(res.agent_invocation_failures)}")
    snap = {'run': run_idx, 'batch_id': res.batch_evaluation_id,
            'batch_arn': res.batch_evaluation_arn, 'status': str(res.status),
            'scores': _aggregate_scores(res), 'agent_perf': _agent_perf_snapshot(),
            'events': _event_rows_for_run(res),
            'agent_runs': list(AGENT_RUNS.values())}
    snap['_result'] = res  # kept in-memory only (not serialised) for the last-run detail cell
    print(f"  run {run_idx} scores: {snap['scores']}")
    return snap


print(f"\nEvaluators : {ALL_EVALUATOR_IDS}  (all SESSION-level)")
print(f"Scenarios  : {len(dataset.scenarios)}   ·   EVAL_K = {EVAL_K}")
print("Starting k-run batch evaluation (invokes the agent per row, then evaluates server-side)...")

# ── Resumable k-run loop ─────────────────────────────────────────────────────
# Each completed run's snapshot is persisted to a resume file the MOMENT it
# finishes, so an interrupted session (kernel / env teardown) loses at most the
# in-flight run, not the whole k=3. Re-executing this cell reloads finished runs
# and only runs the missing indices. The resume file is keyed by (EVAL_ID, EVAL_K)
# so a different layer or k starts clean. Delete it to force a fresh run.
_resume_file = f"../data/eval/results/.resume_metadata_query_kmean_{EVAL_ID}_k{EVAL_K}.json"


def _load_resume() -> list:
    """Return previously-completed run snapshots (without the non-serialisable _result)."""
    if not os.path.exists(_resume_file):
        return []
    try:
        with open(_resume_file) as f:
            data = json.load(f)
        if data.get('eval_id') == EVAL_ID and data.get('eval_k') == EVAL_K:
            return data.get('runs', [])
    except Exception:  # noqa: BLE001 — a corrupt resume file must not block a fresh run
        pass
    return []


def _save_resume(runs: list) -> None:
    """Atomically persist completed run snapshots (strip the live _result object)."""
    serialisable = [{k: v for k, v in r.items() if k != '_result'} for r in runs]
    tmp = _resume_file + '.tmp'
    with open(tmp, 'w') as f:
        json.dump({'eval_id': EVAL_ID, 'eval_k': EVAL_K, 'runs': serialisable},
                  f, indent=2, default=str)
    os.replace(tmp, _resume_file)  # atomic rename — never leaves a half-written file


RUNS = _load_resume()           # per-run snapshots; resumed ones lack the live _result
_done = {r['run'] for r in RUNS}
if _done:
    print(f"↻ Resuming — {len(_done)} run(s) already complete: {sorted(_done)} "
          f"(delete {os.path.basename(_resume_file)} to force a clean run)")

for _i in range(1, EVAL_K + 1):
    if _i in _done:
        print(f"  ✓ run {_i}/{EVAL_K} already complete (resume file) — skipping")
        continue
    RUNS.append(_run_one_batch(_i))
    _save_resume(RUNS)          # persist immediately so a kill after this run keeps it
    print(f"  ↳ run {_i} persisted to {os.path.basename(_resume_file)}")

RUNS.sort(key=lambda r: r['run'])
# batch_result is None when the final run was resumed from disk (its live result
# object isn't serialisable). Cell 17 guards on this and still writes the k-mean
# headline from RUNS.
batch_result = RUNS[-1].get('_result') if RUNS else None

print(f"\n✓ Completed {EVAL_K} run(s)"
      + ("" if batch_result is not None else " (last run resumed from disk — "
         "per-query last-run detail will be skipped; k-mean summary is unaffected)"))


SERVICE_NAME : semantic_layer_dev_metadata_query-6aPZJf6eWO.DEFAULT
LOG GROUPS   : aws/spans, /aws/bedrock-agentcore/runtimes/semantic_layer_dev_metadata_query-6aPZJf6eWO-DEFAULT



Evaluators : ['GoalSuccess_847b3df5-0EcbCECnsF', 'FinalAnswerFaithfulness_847b3df5-4RJPHnAUIW', 'SqlGrounded_847b3df5-ANxPbNGHSa']  (all SESSION-level)
Scenarios  : 16   ·   EVAL_K = 3
Starting k-run batch evaluation (invokes the agent per row, then evaluates server-side)...

=== Run 1/3: query_gt_batch_r1_0d317b38 ===


  OK [gt-row-01-b4e70a87-0080-487d-b6b1-2d2f2df0d659 t0] 20.6s . none


  OK [gt-row-00-4dfdceb9-de16-4e6d-9c14-e1b889bcd761 t0] 24.17s . none


  OK [gt-row-02-a1b85dc3-7f41-4a62-932b-639e3886e116 t0] 32.19s . none


...

  OK [gt-row-13-4f82e82b-c505-4137-8117-48d59b8ea280 t0] 23.84s . none


  OK [gt-row-15-12d57d0d-fb43-4b0b-87ce-cf1e601d608a t0] 18.23s . none


  OK [gt-row-14-5d76b93e-a8c5-4171-b717-f783376878e7 t0] 19.2s . none


  ✓ status: COMPLETED


  run 3 scores: {'FinalAnswerFaithfulness': 0.75, 'SqlGrounded': 1.0, 'GoalSuccess': 0.75}
  ↳ run 3 persisted to .resume_metadata_query_kmean_semantic-rag-multi_table_curated_layer-7e509128_k3.json

✓ Completed 3 run(s)


## 7. Results — aggregate scores, per-query detail, agent cost/latency

Three layers:
1. **Aggregate per-evaluator scores** — from `batch_result.evaluation_results` (in-service).
2. **Per-query evaluator scores** — `fetch_evaluation_events()` reads one event per
   turn-per-evaluator (score, label, explanation) from the batch output log stream, joined
   back to each question by session id.
3. **Per-query agent cost/latency** — from `AGENT_RUNS` (the agent's own tokens / runtimeMs /
   wall-clock; the batch result never carries these).


In [10]:
# @multiturn-cell17  k-run mean scores + last-run per-query detail + preserved JSON dumps.
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# ════════════════════════════════════════════════════════════════════════════
# A. k-run MEAN aggregate scores (the headline) — averaged over RUNS from cell 15.
# ════════════════════════════════════════════════════════════════════════════
import statistics as _stats

_EVAL_ORDER = ['GoalSuccess', 'FinalAnswerFaithfulness',
               'SqlGrounded']


def _mean_std(values: list):
    """Return (mean, std, n) over the non-None numeric values (std=0.0 when n<2)."""
    nums = [v for v in values if isinstance(v, (int, float))]
    if not nums:
        return None, None, 0
    mean = round(sum(nums) / len(nums), 4)
    std = round(_stats.pstdev(nums), 4) if len(nums) > 1 else 0.0
    return mean, std, len(nums)


# Per-evaluator mean ± std across the k runs.
mean_rows = []
for ev_name in _EVAL_ORDER:
    per_run = [r['scores'].get(ev_name) for r in RUNS]
    mean, std, n = _mean_std(per_run)
    mean_rows.append({'Evaluator': ev_name, 'MeanScore': mean, 'Std': std,
                      'Runs': n, 'PerRun': [r['scores'].get(ev_name) for r in RUNS]})
df_mean = pd.DataFrame(mean_rows)

# Mean agent cost/latency across runs.
def _perf_mean(key: str):
    return _mean_std([r['agent_perf'].get(key) for r in RUNS])[0]

agent_perf_mean = {
    'avg_wall_clock_s': _perf_mean('avg_wall_clock_s'),
    'input_tokens': _perf_mean('input_tokens'),
    'output_tokens': _perf_mean('output_tokens'),
    'total_tokens': _perf_mean('total_tokens'),
}

# Per-scenario mean GoalSuccess across runs (handy for downstream matched-subset analysis).
_per_scn = {}
for r in RUNS:
    for e in r['events']:
        if e['evaluator'] == 'GoalSuccess' and isinstance(e['score'], (int, float)):
            _per_scn.setdefault(e['scenario_id'], []).append(e['score'])
per_scenario_goal_mean = {sid: round(sum(v) / len(v), 4) for sid, v in _per_scn.items()}

# Per-scenario agent detail across all k runs (emitted SQL / mean tokens / mean latency).
# Downstream notebook 9 reads this to build its apples-to-apples matched subset (rows where
# BOTH arms emitted SQL) for this arm WITHOUT re-running this notebook's agent. We key off the
# per-run agent_runs snapshots captured in cell 15, folding multi-turn sessions to their FINAL
# turn (the answer-bearing turn) and averaging tokens/latency over the runs that produced them.
def _scn_id_from_session(sess):
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''
_psa = {}
for r in RUNS:
    by_sess = {}
    for rec in r['agent_runs']:
        by_sess.setdefault(rec['scenario_session'], []).append(rec)
    for sess, recs in by_sess.items():
        recs.sort(key=lambda x: x.get('turn_idx', 0))
        final = recs[-1]
        sid = _scn_id_from_session(sess)
        d = _psa.setdefault(sid, {'sql_runs': 0, 'tokens': [], 'latency': [],
                                  'gen_sql': '', 'exec_sql': '', 'rows': None,
                                  'sample': [], 'degraded': None})
        if final.get('agent_sql'):
            d['sql_runs'] += 1
        d['tokens'].append(int((final.get('usage') or {}).get('totalTokens') or 0))
        d['latency'].append(sum(x.get('wall_clock_s') or 0 for x in recs))
        # Keep one representative SQL + result for the evidence record: prefer a
        # run whose final turn actually executed SQL over a degraded/no-SQL one.
        if final.get('agent_sql') and (not d['gen_sql'] or final.get('executed_sql')):
            d['gen_sql'] = final.get('agent_sql', '')
            d['exec_sql'] = final.get('executed_sql', '')
            d['rows'] = final.get('result_row_count')
            d['sample'] = final.get('result_sample', [])
            d['degraded'] = final.get('degraded')
# question text per scenario (the robust cross-notebook join key — gt-row-NN indices differ
# between notebooks because some filter the dataset, but the question text is stable).
_q_by_sid = {s['scenario_id']: (s['turns'][0].get('input', '') if s.get('turns') else '')
             for s in _specs}
# Ground-truth expected SQL + result, keyed by question text, so the evidence
# record pairs the agent's generated SQL/result with what was expected.
_gt_by_q = {row['Natural_Language_Question']: row for row in groundtruth}
_q2sid = {q: sid for sid, q in _q_by_sid.items()}
per_scenario_agent = {}
for sid, d in _psa.items():
    q = _q_by_sid.get(sid, '')
    gt = _gt_by_q.get(q, {})
    gt_res = gt.get('Expected_SQL_Result')
    per_scenario_agent[sid] = {
        'question': q,
        'emitted_sql': d['sql_runs'] > 0,        # emitted SQL in at least one run
        'sql_run_fraction': round(d['sql_runs'] / len(d['tokens']), 4) if d['tokens'] else 0.0,
        'avg_total_tokens': round(sum(d['tokens']) / len(d['tokens'])) if d['tokens'] else 0,
        'avg_wall_clock_s': round(sum(d['latency']) / len(d['latency']), 2) if d['latency'] else None,
        # Evidence: agent's generated/executed SQL + actual result, vs ground truth.
        'generated_sql': d.get('gen_sql', ''),
        'executed_sql': d.get('exec_sql', ''),
        'degraded': d.get('degraded'),
        'result_row_count': d.get('rows'),
        'result_sample': d.get('sample', []),
        'expected_sql': gt.get('Expected_SQL_Query', ''),
        'expected_row_count': len(gt_res) if isinstance(gt_res, list) else None,
        'expected_sample': gt_res[:3] if isinstance(gt_res, list) else gt_res,
    }


print(f"=== k-run MEAN scores over EVAL_K={EVAL_K} run(s) ===")
display(df_mean)
print(f"\nMean agent cost/latency: avg_wall={agent_perf_mean['avg_wall_clock_s']}s  "
      f"total_tokens={agent_perf_mean['total_tokens']}")

# ── Persist the k-run mean summary (the file downstream notebooks 9 & 10 read). ──
kmean_file = f"../data/eval/results/metadata_query_kmean_eval_{timestamp}.json"
kmean = {
    'notebook': '2_metadata_query',
    'arm_label': 'normalized-s3',          # SemanticRAG over the normalized S3 layer
    'agent': 'metadata_query_agent',
    'agent_id': agent_id,
    'runtime_arn': METADATA_QUERY_RUNTIME_ARN,
    'eval_id': EVAL_ID,
    'eval_k': EVAL_K,
    'evaluator_ids': ALL_EVALUATOR_IDS,
    'custom_evaluators': {'GoalSuccess': GOAL_SUCCESS_ID,
                          'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                          'SqlGrounded': SQL_GROUNDED_ID},
    # Mean ± std per evaluator (the headline), plus the per-run scores that produced them.
    'mean_scores': {row['Evaluator']: row['MeanScore'] for row in mean_rows},
    'std_scores': {row['Evaluator']: row['Std'] for row in mean_rows},
    'per_run_scores': [{'run': r['run'], 'batch_id': r['batch_id'],
                        'status': r['status'], 'scores': r['scores'],
                        'agent_perf': r['agent_perf']} for r in RUNS],
    'agent_perf_mean': agent_perf_mean,
    'per_scenario_goal_mean': per_scenario_goal_mean,
    'per_scenario_agent': per_scenario_agent,
}
kmean = _redact_account_ids(kmean)
with open(kmean_file, 'w') as f:
    json.dump(kmean, f, indent=2, default=str)
print(f"\n✓ Wrote k-run mean summary → {kmean_file}")

# ════════════════════════════════════════════════════════════════════════════
# B. Per-query DETAIL for the LAST run (batch_result + AGENT_RUNS point at run k).
#    Preserves the metadata_query_batch_eval_*.json file notebook 4 globs for.
# ════════════════════════════════════════════════════════════════════════════
try:
    from agents.shared.eval_multiturn import group_runs_by_session
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import group_runs_by_session

# GUARD: batch_result is None when the final k-run was RESUMED from disk (the
# live BatchEvaluation object isn't serialisable). The k-mean headline + kmean_file
# above are already complete; skip the last-run drill-in rather than deref None.
# Delete the .resume_*.json file and re-run the batch cell for a fresh in-memory run.
if batch_result is None:
    print("\n⚠ Last-run detail SKIPPED — final run resumed from disk (no live batch "
          "result). k-run mean summary above is complete and saved.")
    combined_file = None
else:
    print(f"\n── Last-run detail — Batch ID: {batch_result.batch_evaluation_id} "
          f"(status {batch_result.status}) ──")

    # ── 1. Aggregate per-evaluator scores (last run) ─────────────────────────────
    agg_rows = []
    ev = batch_result.evaluation_results
    if ev is not None:
        print(f"Sessions: completed={ev.number_of_sessions_completed} "
              f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
        for es in (ev.evaluator_summaries or []):
            stats = es.statistics
            avg = (f"{stats.average_score:.3f}"
                   if stats and stats.average_score is not None else None)
            agg_rows.append({'Evaluator': es.evaluator_id or 'unknown', 'AvgScore': avg,
                             'Evaluated': es.total_evaluated or 0, 'Failed': es.total_failed or 0})
    else:
        print("⚠ No aggregate evaluation_results — check job status / spans.")
    df_agg = pd.DataFrame(agg_rows)

    grouped = group_runs_by_session(AGENT_RUNS)

    # ── 2. Per-query evaluator events (last run) — reuse the helper from cell 15 ──
    event_rows = _event_rows_for_run(batch_result)
    # Re-add the explanation field for the displayed detail table.
    def _f(e, key):
        return e[key] if key in e else (e.get('attributes', {}) or {}).get(key)
    _explan = {}
    for e in _fetch_events(batch_result):
        sess = _f(e, 'session.id') or _f(e, 'gen_ai.session.id')
        name = _f(e, 'gen_ai.evaluation.name')
        _explan[(_scenario_id_for_session(sess), _EVAL_NAME.get(name, name))] = \
            (str(_f(e, 'gen_ai.evaluation.explanation') or ''))
    for row in event_rows:
        row['explanation'] = _explan.get((row['scenario_id'], row['evaluator']), '')
    df_events = pd.DataFrame(event_rows)
    print(f"Per-query evaluator events (last run): {len(df_events)}")

    # ── 3. Per-turn trajectory table + clarification summary (last run) ──────────
    turn_rows = []
    for sess, turns in grouped.items():
        sid = _scenario_id_for_session(sess)
        for t in turns:
            turn_rows.append({
                'scenario_id': sid, 'turn': t.get('turn_idx'),
                'message': (t.get('message') or ''), 'clarified': t.get('clarified'),
                'has_sql': bool(t.get('agent_sql')), 'wall_s': t.get('wall_clock_s'),
                'error': t.get('invoke_error'),
            })
    df_agent = pd.DataFrame(turn_rows)
    if not df_agent.empty:
        df_agent = df_agent.sort_values(['scenario_id', 'turn']).reset_index(drop=True)

    print("\n── Trajectory summary (last run, per scenario) ──")
    for sess, turns in grouped.items():
        sid = _scenario_id_for_session(sess)
        clar = [t.get('turn_idx') for t in turns if t.get('clarified')]
        final_sql = bool(turns[-1].get('agent_sql')) if turns else False
        print(f"  {sid}: {len(turns)} turn(s) · clarified_on={clar or 'none'} · "
              f"re_clarified={len(clar) > 1} · final_sql={final_sql}")

    # ── 3b. Correct-tier assertion (Step 5: intent router + provenance) ──────────
    _spec_by_id = {s['scenario_id']: s for s in _specs}
    tier_rows = []
    for sess, turns in grouped.items():
        sid = _scenario_id_for_session(sess)
        spec = _spec_by_id.get(sid, {})
        expected = spec.get('expected_tier', 'semantic_sql')
        final = turns[-1] if turns else {}
        prov = final.get('provenance') or {}
        actual = prov.get('tier') or ('semantic_sql' if final.get('agent_sql') else 'unknown')
        tier_rows.append({
            'scenario_id': sid, 'question': (turns[0]['message'] if turns else '')[:70],
            'expected_tier': expected, 'actual_tier': actual, 'match': expected == actual,
        })
    df_tier = pd.DataFrame(tier_rows)
    _tier_correct = sum(1 for r in tier_rows if r['match'])
    _tier_total = len(tier_rows)
    print(f"\n── Correct-tier routing (last run): {_tier_correct}/{_tier_total} "
          f"({100*_tier_correct//_tier_total if _tier_total else 0}%) ──")
    for r in tier_rows:
        print(f"  {'OK' if r['match'] else 'XX'} {r['scenario_id']}: "
              f"expected={r['expected_tier']} actual={r['actual_tier']}")
    if not df_tier.empty:
        display(df_tier)

    # ── 4. Agent cost/latency (last run) ─────────────────────────────────────────
    _lat = [r['wall_clock_s'] for r in AGENT_RUNS.values() if r.get('wall_clock_s') is not None]
    def _usage_sum(key):
        return sum(int((r.get('usage') or {}).get(key) or 0) for r in AGENT_RUNS.values())
    agent_perf = {
        'turns': len(AGENT_RUNS), 'sessions': len(grouped),
        'avg_wall_clock_s': round(sum(_lat) / len(_lat), 2) if _lat else None,
        'input_tokens': _usage_sum('inputTokens'), 'output_tokens': _usage_sum('outputTokens'),
        'total_tokens': _usage_sum('totalTokens'),
    }
    print(f"\n── Query agent cost/latency (last run) ──")
    print(f"  Avg wall-clock : {agent_perf['avg_wall_clock_s']}s  ·  tokens: {agent_perf['total_tokens']}")

    # ── Persist the last-run detail (filename pattern + keys notebook 4 reads). ──
    combined_file = f"../data/eval/results/metadata_query_batch_eval_{timestamp}.json"
    combined = {
        'agent_id': agent_id, 'runtime_arn': METADATA_QUERY_RUNTIME_ARN, 'eval_id': EVAL_ID,
        'eval_k': EVAL_K, 'kmean_file': os.path.basename(kmean_file),
        'batch_evaluation_id': batch_result.batch_evaluation_id,
        'batch_evaluation_arn': batch_result.batch_evaluation_arn,
        'status': str(batch_result.status), 'evaluator_ids': ALL_EVALUATOR_IDS,
        'custom_evaluators': {'GoalSuccess': GOAL_SUCCESS_ID,
                              'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                              'SqlGrounded': SQL_GROUNDED_ID},
        'aggregate_summaries': agg_rows, 'per_query_events': event_rows,
        'agent_runs': list(AGENT_RUNS.values()), 'agent_performance': agent_perf,
        'correct_tier': {'rows': tier_rows, 'correct': _tier_correct, 'total': _tier_total},
    }
    combined = _redact_account_ids(combined)
    with open(combined_file, 'w') as f:
        json.dump(combined, f, indent=2, default=str)
    print(f"✓ Wrote last-run detail → {combined_file}")

    print("\n=== Aggregate per-evaluator scores (last run) ===")
    display(df_agg)
    print("=== Per-query evaluator scores (last run) ===")
    display(df_events)
    print("=== Per-turn trajectory (last run) ===")
    display(df_agent)


=== k-run MEAN scores over EVAL_K=3 run(s) ===


,Evaluator,MeanScore,Std,Runs,PerRun
0,GoalSuccess,0.75,0.0,3,"[0.75, 0.75, 0.75]"
1,FinalAnswerFaithfulness,0.75,0.0,3,"[0.75, 0.75, 0.75]"
2,SqlGrounded,1.00,0.0,3,"[1.0, 1.0, 1.0]"



Mean agent cost/latency: avg_wall=26.2933s  total_tokens=1015687.3333

✓ Wrote k-run mean summary → ../data/eval/results/metadata_query_kmean_eval_20260701_105036.json

── Last-run detail — Batch ID: query_gt_batch_r3_cc5691b4-3e866f984d (status COMPLETED) ──
Sessions: completed=16 failed=0 total=16


Per-query evaluator events (last run): 48

── Trajectory summary (last run, per scenario) ──
  gt-row-00: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=False
  gt-row-01: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=False
  gt-row-04: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=False
  gt-row-03: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  gt-row-02: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=False
  gt-row-05: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  gt-row-06: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  mt-parties-clarify: 2 turn(s) · clarified_on=[0] · re_clarified=False · final_sql=True
  gt-row-08: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  gt-row-07: 1 turn(s) · clarified_on=none · re_clarified=False · final_sql=True
  mt-stable-options: 3 turn(s) · clarified_on=[0, 1] · re_clarified=True · final_sql=

,scenario_id,question,expected_tier,actual_tier,match
0,gt-row-00,Show me policies where the insured party is also the policyholder.,semantic_sql,semantic_sql,True
1,gt-row-01,"For each rider, who are the insured participants and what are their ro",semantic_sql,semantic_sql,True
2,gt-row-04,"For policies that have a payout schedule, show the policyholder's name",semantic_sql,semantic_sql,True
3,gt-row-03,"What is the total market value of all active holdings, grouped by part",semantic_sql,semantic_sql,True
4,gt-row-02,List the top 5 most common party types and their human-readable descri,semantic_sql,semantic_sql,True
5,gt-row-05,How many parties are there?,semantic_sql,semantic_sql,True
6,gt-row-06,List the top 10 coverage products by name.,semantic_sql,semantic_sql,True
7,mt-parties-clarify,How many are there?,semantic_sql,semantic_sql,True
8,gt-row-08,What was the total financial-activity amount per month in 2024?,semantic_sql,semantic_sql,True
9,gt-row-07,"Show the top 10 parties by total holding market value, including the i",semantic_sql,semantic_sql,True


...

,Evaluator,AvgScore,Evaluated,Failed
0,FinalAnswerFaithfulness_847b3df5-4RJPHnAUIW,0.750,16,0
1,SqlGrounded_847b3df5-ANxPbNGHSa,1.000,16,0
2,GoalSuccess_847b3df5-0EcbCECnsF,0.750,16,0


=== Per-query evaluator scores (last run) ===


,scenario_id,question,evaluator,score,label,explanation
0,gt-row-08,What was the total financial-activity amount per month in 2024?,FinalAnswerFaithfulness,1.0,pass,"The agent's final answer reports March 2024 $70,895.29, May 2024 $20,242.90, and October 2024 $13,991.93, matching exactly the expected answer's three months and totals. No contradictions or omissions of required data."
1,gt-row-08,What was the total financial-activity amount per month in 2024?,SqlGrounded,1.0,pass,"The executed SQL selects transaction_date, transaction_amount, and is_deleted from normalized.financial_activity. All three columns (transaction_date, transaction_amount, is_deleted) plus the table normalized.financial_activity are present in the retrieved schema context. SUBSTR, SUM, CAST are SQL builtins, not schema elements. No hallucinated tables/columns/joins are used."
2,gt-row-08,What was the total financial-activity amount per month in 2024?,GoalSuccess,1.0,pass,"The final-answer record reports total financial-activity amounts per month in 2024: $70,895.29 in March, $20,242.90 in May, and $13,991.93 in October, matching the SQL query results exactly. This matches the expected answer's three months and totals precisely. No clarification was needed, and the answer is substantive and correct."
3,mt-no-spurious-clarify,How many parties are there?,GoalSuccess,1.0,pass,"The agent did not ask a clarifying question; it directly interpreted 'parties' as normalized.party table. It executed a COUNT query (SELECT COUNT(party_id) FROM normalized.party) on the first turn and returned a numeric answer (15). The final answer states 'The query returned a count of 15 parties', matching the expected answer 'There are 15 parties.' All three assertions are satisfied."
4,mt-no-spurious-clarify,How many parties are there?,FinalAnswerFaithfulness,1.0,pass,"The agent executed a COUNT query on normalized.party and returned 15 parties, matching the expected final answer of 'There are 15 parties.' The answer is factually consistent with the ground truth."
5,mt-no-spurious-clarify,How many parties are there?,SqlGrounded,1.0,pass,"The executed SQL is `SELECT COUNT(party_id) FROM normalized.party`. The table normalized.party is referenced throughout the schema context as the join target (e.g., party.party_id) though not explicitly listed in the 'tables' array; it's clearly part of the allowed schema as the central party table referenced by all other tables' joins. The column party_id is explicitly documented as party.party_id in multiple join descriptions. Thus both the table and column are grounded in the retrieved schema context."
6,gt-row-06,List the top 10 coverage products by name.,FinalAnswerFaithfulness,1.0,pass,"The agent's final answer lists 5 active coverage products: Accidental Death, Base Life, Spouse Rider, Term Rider, and Child Rider, alphabetically ordered, matching the expected answer's list and count exactly (same 5 names, same alphabetical ordering). The categories mentioned are consistent with the retrieved data and don't contradict the expected answer. This matches the ground truth in entities, count, and conclusion."
7,gt-row-06,List the top 10 coverage products by name.,GoalSuccess,1.0,pass,"The final-answer record's output reports 5 active coverage products, listed alphabetically: Accidental Death, Base Life, Spouse Rider, Term Rider (Death Benefit) and Child Rider (Living Benefit). This matches the expected answer stating there are 5 active products total, alphabetically: Accidental Death, Base Life, Child Rider, Spouse Rider, Term Rider. The set and count match exactly, satisfying the assertion despite minor ordering variance in the narrative text (the underlying data and count are correct)."
8,gt-row-06,List the top 10 coverage products by name.,SqlGrounded,1.0,pass,"The executed SQL: SELECT product_name, coverage_category FROM normalized.coverage_product WHERE is_deleted = false ORDER BY product_name LIMIT 10. Checking against the retrieved schema 

=== Per-turn trajectory (last run) ===


,scenario_id,turn,message,clarified,has_sql,wall_s,error
0,gt-row-00,0,Show me policies where the insured party is also the policyholder.,False,False,18.27,None
1,gt-row-01,0,"For each rider, who are the insured participants and what are their roles?",False,False,23.76,None
2,gt-row-02,0,List the top 5 most common party types and their human-readable descriptions.,False,False,59.85,None
3,gt-row-03,0,"What is the total market value of all active holdings, grouped by party?",False,True,35.16,None
4,gt-row-04,0,"For policies that have a payout schedule, show the policyholder's name, the payout frequency, and the projected next-payout amount.",False,False,17.13,None
5,gt-row-05,0,How many parties are there?,False,True,30.10,None
6,gt-row-06,0,List the top 10 coverage products by name.,False,True,22.15,None
7,gt-row-07,0,"Show the top 10 parties by total holding market value, including the investment product names they hold.",False,True,39.85,None
8,gt-row-08,0,What was the total financial-activity amount per month in 2024?,False,True,24.15,None
9,gt-row-13,0,What are some common metrics that could be calculated with this data?,False,False,23.84,None


## 8. Store IDs for Downstream Notebooks

Save the batch evaluation id, the per-query session ids, and the agent id so downstream
notebooks (e.g. notebook 3, online evaluation) can reference this run.

In [11]:
query_batch_id = batch_result.batch_evaluation_id
query_eval_session_ids = list(AGENT_RUNS.keys())
metadata_query_agent_id = agent_id
%store query_batch_id
%store query_eval_session_ids
%store metadata_query_agent_id

print("✓ Stored for downstream notebooks:")
print(f"  query_batch_id        = {query_batch_id}")
print(f"  query_eval_session_ids= {len(query_eval_session_ids)} session(s)")
print(f"  metadata_query_agent_id = {metadata_query_agent_id}")
print(f"\n  Combined results: {combined_file}")


Stored 'query_batch_id' (str)
Stored 'query_eval_session_ids' (list)
Stored 'metadata_query_agent_id' (str)
✓ Stored for downstream notebooks:
  query_batch_id        = query_gt_batch_r3_cc5691b4-3e866f984d
  query_eval_session_ids= 19 session(s)
  metadata_query_agent_id = semantic_layer_dev_metadata_query-6aPZJf6eWO

  Combined results: ../data/eval/results/metadata_query_batch_eval_20260701_105036.json


## Summary

This notebook evaluates the deployed Metadata Query Agent **entirely server-side** via
AgentCore Batch Evaluations. The only client-side work is invoking the agent once per
ground-truth turn (required to produce spans) and reading its response for cost/latency.

### Pipeline
1. **Custom evaluators** (LLM-as-Judge, no Lambda) — all **SESSION-level**:
   `GoalSuccess`, `FinalAnswerFaithfulness`, and `SqlGrounded`, created with
   `create_evaluator` and scored in-service.
2. **Dataset** — one `PredefinedScenario` per ground-truth row (per-conversation sessions);
   multi-turn rows carry several `Turn`s. `expected_response` is set on the **final** turn
   only (TRACE ground truth), and the expected answer is **also** threaded into `assertions`
   so the SESSION `FinalAnswerFaithfulness` judge can read it. Each scenario carries
   `expected_trajectory` and `assertions`.
3. **agent_invoker** — drives the chat-stream path (reads + persists DDB history) so turn N+1
   resolves turn N's clarification; records the agent's own tokens / `runtimeMs` / wall-clock
   per turn into `AGENT_RUNS`.
4. **BatchEvaluationRunner** — one `StartBatchEvaluation` job over the SESSION-only evaluator
   set.
5. **Results** — aggregate per-evaluator scores, **per-query** scores via
   `fetch_evaluation_events`, and **per-turn** agent cost/latency + trajectory.

### Why SESSION-level evaluators (the multi-turn fix)
This agent is multi-turn: it may answer in one turn, or ask a **clarifying question** first
and answer on a later turn. A SESSION-level judge reads the whole conversation via `{context}`
and scores the **final** answer once — the correct granularity for a multi-turn agent (a
clarify-then-answer conversation is judged on its destination, not on each intermediate turn).

So SESSION-level is chosen for **granularity**, not to dodge per-turn span gaps: answer quality
for a multi-turn conversation is a once-per-conversation question, and `{expected_response}` is
a TRACE-only placeholder (the expected answer is threaded into `{assertions}` for the SESSION
`FinalAnswerFaithfulness` judge). **Trade-off:** the per-turn `Builtin.Correctness` /
`Builtin.Helpfulness` signal is not collected; the per-turn trajectory table (clarified /
has_sql / latency) covers per-turn behaviour instead.